In [1]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

sys.path.append('../reconstruction')
sys.path.append('./reconstruction')
from util import Population

from standardize_align_new import Standardizer, SingleTransformerEncoderStandardizer

from models import base_CNN, two_CNN, var_CNN, base_RNN, var_RNN, base_TransformerEncoder

generalized align


In [2]:
def print_weights(model):
    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f'Layer: {name} - Weights')
            print(param)
        elif 'bias' in name:
            print(f'Layer: {name} - Biases')
            print(param)

def print_pairs(model, blackbox):
    for (name1, param1), (name2, param2) in zip(model.named_parameters(), blackbox.named_parameters()):
        if 'weight' in name1:
            print(f'Layer: {name1} - Reconstructed Weights')
            print(param1)
            print(f'Layer: {name2} - Blackbox Weights')
            print(param2)
        elif 'bias' in name1:
            print(f'Layer: {name1} - Reconstructed Biases')
            print(param1)
            print(f'Layer: {name2} - Blackbox Biases')
            print(param2)

In [3]:
#RNN

blackbox_dict_path = "rnn/seed_31_RNNx28_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_var_RNN_28/black_box.pt"
blackbox_og_params_dict_path = "rnn/seed_31_RNNx28_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_var_RNN_28/original_params_black_box.pt"
final_population_dict_path = "rnn/seed_31_RNNx28_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_var_RNN_28/population_iteration_54.pt"

best_model_index = 1 #needs to be manually inspected from log file

blackbox_dict = torch.load(blackbox_dict_path)
blackbox_original_params_dict = torch.load(blackbox_og_params_dict_path)
final_population_dict = torch.load(final_population_dict_path)

blackbox = var_RNN(28, [28]) #input_size, layer_configs
blackbox.load_state_dict(blackbox_dict)

subs = [var_RNN(28, [28]) for i in range(10)]
print(final_population_dict.keys())
final_population = Population(subs=subs)
final_population.load_state_dict(final_population_dict)
final_population.best = best_model_index
best_model = final_population.subs[best_model_index]

final_population.evaluate(blackbox, model_type='rnn')

odict_keys(['subs.0.layers.0.weight_ih_l0', 'subs.0.layers.0.weight_hh_l0', 'subs.0.layers.0.bias_ih_l0', 'subs.0.layers.0.bias_hh_l0', 'subs.0.layers.1.weight', 'subs.0.layers.1.bias', 'subs.1.layers.0.weight_ih_l0', 'subs.1.layers.0.weight_hh_l0', 'subs.1.layers.0.bias_ih_l0', 'subs.1.layers.0.bias_hh_l0', 'subs.1.layers.1.weight', 'subs.1.layers.1.bias', 'subs.2.layers.0.weight_ih_l0', 'subs.2.layers.0.weight_hh_l0', 'subs.2.layers.0.bias_ih_l0', 'subs.2.layers.0.bias_hh_l0', 'subs.2.layers.1.weight', 'subs.2.layers.1.bias', 'subs.3.layers.0.weight_ih_l0', 'subs.3.layers.0.weight_hh_l0', 'subs.3.layers.0.bias_ih_l0', 'subs.3.layers.0.bias_hh_l0', 'subs.3.layers.1.weight', 'subs.3.layers.1.bias', 'subs.4.layers.0.weight_ih_l0', 'subs.4.layers.0.weight_hh_l0', 'subs.4.layers.0.bias_ih_l0', 'subs.4.layers.0.bias_hh_l0', 'subs.4.layers.1.weight', 'subs.4.layers.1.bias', 'subs.5.layers.0.weight_ih_l0', 'subs.5.layers.0.weight_hh_l0', 'subs.5.layers.0.bias_ih_l0', 'subs.5.layers.0.bias_hh

model_type:  rnn


In [4]:
#transformer

blackbox_dict_path = "transformer/seed_0_transx64_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_test3/black_box.pt"
blackbox_og_params_dict_path = "transformer/seed_0_transx64_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_test3/original_params_black_box.pt"
final_population_dict_path = "transformer/seed_0_transx64_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_test3/population_iteration_54.pt"

best_model_index = 1

blackbox_dict = torch.load(blackbox_dict_path)
blackbox_original_params_dict = torch.load(blackbox_og_params_dict_path)
final_population_dict = torch.load(final_population_dict_path)

blackbox = base_TransformerEncoder(28, [64]) #d_model, layer_configs (size of linear layer)
blackbox.load_state_dict(blackbox_dict)
 
subs = [base_TransformerEncoder(28, [64]) for i in range(10)]
print(final_population_dict.keys())
final_population = Population(subs=subs)
final_population.load_state_dict(final_population_dict)
final_population.best = best_model_index
best_model = final_population.subs[best_model_index]

final_population.evaluate(blackbox, model_type='trans')

odict_keys(['subs.0.encoder_layer.self_attn.in_proj_weight', 'subs.0.encoder_layer.self_attn.in_proj_bias', 'subs.0.encoder_layer.self_attn.out_proj.weight', 'subs.0.encoder_layer.self_attn.out_proj.bias', 'subs.0.encoder_layer.linear1.weight', 'subs.0.encoder_layer.linear1.bias', 'subs.0.encoder_layer.linear2.weight', 'subs.0.encoder_layer.linear2.bias', 'subs.0.encoder_layer.norm1.weight', 'subs.0.encoder_layer.norm1.bias', 'subs.0.encoder_layer.norm2.weight', 'subs.0.encoder_layer.norm2.bias', 'subs.0.linear.weight', 'subs.0.linear.bias', 'subs.1.encoder_layer.self_attn.in_proj_weight', 'subs.1.encoder_layer.self_attn.in_proj_bias', 'subs.1.encoder_layer.self_attn.out_proj.weight', 'subs.1.encoder_layer.self_attn.out_proj.bias', 'subs.1.encoder_layer.linear1.weight', 'subs.1.encoder_layer.linear1.bias', 'subs.1.encoder_layer.linear2.weight', 'subs.1.encoder_layer.linear2.bias', 'subs.1.encoder_layer.norm1.weight', 'subs.1.encoder_layer.norm1.bias', 'subs.1.encoder_layer.norm2.weight

model_type:  trans
